# Build Verdict

Last task of the build job (`build_verdict`, `run_if: ALL_DONE`, depends on every runnable wave's `w<wave>_gate`). It fails the run when any wave gate failed, so notifications and the run list show the failure.

Why a separate task: the gate notebook must not raise. A failed gate task makes the next wave `UPSTREAM_FAILED`, and that wave's `run_if: ALL_DONE` integrator and gate would still run (Jobs treats Upstream failed as failed, and ALL_DONE runs regardless). The If/else `gate_check` keeps excluding later waves at zero compute; this task turns the outcome into the run result.

Per runnable wave (latest attempt of `w<wave>_gate` in this run, from `runs/get`):
* not in the run (`only` subset): ignored
* `EXCLUDED` (an earlier wave failed): ignored
* `SUCCESS`: `wave_<wave>_gate_result.json` decides; a missing file fails
* any other state (`FAILED`, `TIMEDOUT`, `UPSTREAM_FAILED`, ...): fails

If the Jobs API cannot be read, only the result files decide. The verdict is printed (`BUILD_PASSED` / `BUILD_FAILED`) and written to `build_verdict.json` in the run's status directory. Runs started with `only` must include `build_verdict` to get a verdict.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("run_id", "", "Job run id")
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")

run_id = dbutils.widgets.get("run_id").strip()
build_root = dbutils.widgets.get("build_root").strip()
manifest_path = dbutils.widgets.get("manifest_path").strip()

if not run_id or not build_root or not manifest_path:
    raise ValueError(f"run_id, build_root and manifest_path are required (got {run_id!r}, {build_root!r}, {manifest_path!r})")

print(f"run_id={run_id}")

In [0]:
import json
import os
import sys

# Helper modules sit next to this notebook (cwd for workspace notebooks).
sys.path.insert(0, os.getcwd())
from gate_helpers import build_verdict, gate_result_path, latest_task_states, verdict_message
from jobs_api import get_run
from waves_manifest import runnable_waves
from databricks.sdk import WorkspaceClient

with open(manifest_path) as f:
    manifest = json.load(f)

status_dir = os.path.join(build_root, manifest["status_dir"], run_id)
wave_ids = [str(wv["wave_id"]) for wv in runnable_waves(manifest)]

results = {}
for wid in wave_ids:
    p = gate_result_path(status_dir, wid)
    if os.path.isfile(p):
        try:
            with open(p) as f:
                results[wid] = json.load(f)
        except (OSError, ValueError) as e:
            results[wid] = {"status": "fail", "problems": [f"gate result file unreadable: {p} ({type(e).__name__})"]}

try:
    task_states = latest_task_states(get_run(WorkspaceClient().api_client, run_id).get("tasks"))
except Exception as e:  # interactive run with a fake run_id, or an API error
    print(f"WARN: could not read this run's task states ({type(e).__name__}: {str(e)[:200]}); "
          f"deciding from the gate result files only")
    task_states = None

verdict = build_verdict(wave_ids, task_states, results)
message = verdict_message(verdict)
print(json.dumps(verdict, indent=2, default=str))
print(message)
if verdict["checked"] == 0:
    print("NOTE: no wave gate was evaluated in this run (all excluded or not in the run)")

try:
    os.makedirs(status_dir, exist_ok=True)
    with open(os.path.join(status_dir, "build_verdict.json"), "w") as f:
        json.dump({**verdict, "message": message}, f, indent=2, default=str)
except OSError as e:
    print(f"WARN: could not write build_verdict.json: {e}")

if not verdict["passed"]:
    raise RuntimeError(message)

dbutils.notebook.exit(message)